<a href="https://colab.research.google.com/github/aferenaz/MLOPSII-tp3/blob/main/mini_tp3_colab_Ferenaz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-TP 3 — Sirve tu modelo por gRPC (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Expón **tu propio modelo** (el mismo que serviste por REST en la Sesión 1) como un servicio **gRPC**. Este notebook es un *starter*: completa las celdas marcadas con `# TODO`.

**Qué debes entregar**
1. Un `scoring.proto` con un servicio para tu modelo (mensajes de entrada y salida **tipados**).
2. Los *stubs* generados y un **servidor** que cargue tu modelo **una sola vez**.
3. Un **cliente** que llame al servicio (**unary**).
4. Un método de **server-streaming** que puntúe un lote.
5. Una **comparación de latencia** contra tu endpoint REST de la Sesión 1, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; que el `.proto` tipe entrada y salida; que funcionen unary y streaming; y la reflexión gRPC vs REST.

> Apóyate en `grpc_tutorial.ipynb`, que hace todo esto paso a paso. Entorno con uv:
> `uv add grpcio grpcio-tools scikit-learn joblib requests` (y lo que use tu modelo).

---

**Modelo servido:**  forecasting de demanda diaria para **Gout** (cadena sin TACC,
9 sucursales, 96 SKUs). `RandomForestRegressor` sobre 11 features, con el mismo contrato que el
endpoint REST `/v1/predict` de la Sesión 1: `gout-demanda-rf`, versión `1.0.0`.

**Versión para Google Colab:** corré las celdas en orden (**Entorno de ejecución → Ejecutar todas**).
Todo pasa dentro del notebook: el servidor gRPC y el REST corren en hilos de fondo, sin terminales.


In [ ]:
# Dependencias. En Colab se actualiza protobuf junto con grpcio-tools: el código que genera
# protoc exige una versión de protobuf igual o más nueva que la suya.
# (Equivalente uv: uv add grpcio grpcio-tools protobuf scikit-learn joblib requests fastapi "uvicorn[standard]")
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "grpcio", "grpcio-tools", "protobuf"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "joblib", "requests", "fastapi", "uvicorn"])
print("Listo.")


Listo.


## 1. Tu modelo

Cargo el artefacto de mi Mini-TP 1 (`model/model.joblib`: un `dict` con el modelo, la lista de
features, el nombre y la versión). Si el archivo no está (por ejemplo, en un Colab nuevo), lo
regenero con **exactamente** la receta de `train.py` del Mini-TP 1: mismos datos sintéticos del
dominio, misma semilla y mismos hiperparámetros, así que el modelo es el mismo.


In [ ]:
import os, time
from datetime import datetime, timezone
import joblib
import numpy as np

ARTIFACT_PATH = "model/model.joblib"
FEATURES = [
    "sucursal_id", "sku_id", "dia_semana", "mes", "es_finde", "es_feriado",
    "en_promocion", "precio_unitario", "demanda_lag_1", "demanda_lag_7", "media_movil_7",
]


def entrenar_como_mini_tp1():
    """La receta de train.py del Mini-TP 1: datos sintéticos de Gout + RandomForestRegressor."""
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score

    rng = np.random.default_rng(42)
    n = 8_000
    sucursal_id = rng.integers(1, 10, n)
    sku_id = rng.integers(1, 97, n)
    dia_semana = rng.integers(0, 7, n)
    mes = rng.integers(1, 13, n)
    es_finde = (dia_semana >= 5).astype(int)
    es_feriado = (rng.random(n) < 0.05).astype(int)
    en_promocion = (rng.random(n) < 0.20).astype(int)
    precio_unitario = rng.uniform(800, 3500, n)
    demanda_lag_1 = rng.uniform(0, 120, n)
    demanda_lag_7 = rng.uniform(0, 120, n)
    media_movil_7 = np.clip((demanda_lag_1 + demanda_lag_7) / 2 + rng.normal(0, 8, n), 0, None)
    base = (18 + 0.35 * demanda_lag_1 + 0.30 * demanda_lag_7 + 0.25 * media_movil_7
            + 14 * es_finde + 20 * es_feriado + 12 * en_promocion - 0.004 * precio_unitario
            + 4 * np.sin(2 * np.pi * mes / 12) + (sku_id % 7))
    y = np.clip(base + rng.normal(0, 6, n), 0, None)
    X = np.column_stack([sucursal_id, sku_id, dia_semana, mes, es_finde, es_feriado,
                         en_promocion, precio_unitario, demanda_lag_1, demanda_lag_7, media_movil_7])

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    m = RandomForestRegressor(n_estimators=120, max_depth=10, min_samples_leaf=10,
                              n_jobs=-1, random_state=42).fit(X_tr, y_tr)
    pred = m.predict(X_te)
    return {"model": m, "features": FEATURES, "name": "gout-demanda-rf", "version": "1.0.0",
            "trained_at": datetime.now(timezone.utc).isoformat(),
            "metrics": {"mae": round(mean_absolute_error(y_te, pred), 2),
                        "r2": round(r2_score(y_te, pred), 3)}}


if not os.path.exists(ARTIFACT_PATH):
    print("No encontré model/model.joblib: lo regenero con la receta del Mini-TP 1...")
    os.makedirs("model", exist_ok=True)
    joblib.dump(entrenar_como_mini_tp1(), ARTIFACT_PATH)

# Se carga UNA sola vez: lo usan el servidor gRPC y el REST.
artefacto = joblib.load(ARTIFACT_PATH)
modelo = artefacto["model"]
modelo.n_jobs = 1          # en serving, una predicción por llamada: sin pool de procesos
MODEL_NAME, MODEL_VERSION = artefacto["name"], artefacto["version"]
N_FEATURES = len(artefacto["features"])
assert artefacto["features"] == FEATURES, "las features del artefacto no coinciden con el contrato"
print(f"Modelo cargado: {MODEL_NAME} v{MODEL_VERSION} | n_features = {N_FEATURES} | métricas {artefacto['metrics']}")


No encontré model/model.joblib: lo regenero con la receta del Mini-TP 1...
Modelo cargado: gout-demanda-rf v1.0.0 | n_features = 11 | métricas {'mae': 6.26, 'r2': 0.893}


## 2. El contrato `.proto`

El contrato replica el de Pydantic de la Sesión 1, pero en Protocol Buffers: **un campo con nombre y
tipo por feature** (no un `repeated double` genérico), así el contrato dice qué es cada valor.

- `Predict` (**unary**): un caso → una predicción.
- `PredictLote` (**server-streaming**): un lote de casos → una predicción por caso, que el cliente va
  recibiendo a medida que llega.
- `Health`: el equivalente al `/health` de REST.

proto3 no tiene rangos (`ge`/`le` de Pydantic): la validación va en el servidor, que responde
`INVALID_ARGUMENT`, el equivalente gRPC del 422.


In [ ]:
%%writefile scoring.proto
syntax = "proto3";
package scoring;

// Entrada: las 11 features del modelo de demanda de Gout, cada una con su tipo.
message Features {
  int32  sucursal_id     = 1;   // 1..9
  int32  sku_id          = 2;   // 1..96
  int32  dia_semana      = 3;   // 0..6 (0 = lunes)
  int32  mes             = 4;   // 1..12
  bool   es_finde        = 5;
  bool   es_feriado      = 6;
  bool   en_promocion    = 7;
  double precio_unitario = 8;   // ARS, > 0
  double demanda_lag_1   = 9;   // >= 0
  double demanda_lag_7   = 10;  // >= 0
  double media_movil_7   = 11;  // >= 0
}

// Salida: la misma respuesta que /v1/predict de la Sesión 1.
message Prediction {
  double demanda_estimada = 1;
  string unidad           = 2;
  string modelo_nombre    = 3;
  string modelo_version   = 4;
  int32  indice           = 5;  // posición del caso dentro del lote (en el streaming)
}

// Un lote de casos para puntuar en streaming.
message Lote {
  repeated Features casos = 1;
}

message HealthRequest {}
message HealthResponse {
  string status         = 1;
  bool   modelo_cargado = 2;
  string modelo_nombre  = 3;
  string modelo_version = 4;
  int32  n_features     = 5;
}

service Scoring {
  rpc Predict(Features) returns (Prediction);              // unary: 1 -> 1
  rpc PredictLote(Lote) returns (stream Prediction);       // server streaming: 1 lote -> N predicciones
  rpc Health(HealthRequest) returns (HealthResponse);
}


Writing scoring.proto


## 3. Genera los *stubs*

In [ ]:
import sys, subprocess, os, importlib
subprocess.run([sys.executable, "-m", "grpc_tools.protoc",
                "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto"], check=True)
print("Generados:", sorted(f for f in os.listdir('.') if f.startswith('scoring_pb2')))

# Chequeo de versiones: si protobuf quedó viejo, el import falla con un VersionError.
try:
    import scoring_pb2
except Exception as e:
    raise RuntimeError(
        "protobuf no coincide con el código generado. Hacé Entorno de ejecución -> Reiniciar sesión "
        "y volvé a correr todo desde arriba (la celda de dependencias ya dejó protobuf actualizado)."
    ) from e
import google.protobuf
print("protobuf", google.protobuf.__version__, "| stubs ok")


Generados: ['scoring_pb2.py', 'scoring_pb2_grpc.py']
protobuf 7.36.2 | stubs ok


## 4. El servidor

El modelo ya está cargado (sección 1): el servidor sólo lo usa, nunca lo carga dentro de un método.
Cada método valida la entrada con las mismas reglas que el contrato Pydantic de la Sesión 1.


In [ ]:
import grpc
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc
importlib.reload(scoring_pb2); importlib.reload(scoring_pb2_grpc)   # por si se regeneró el .proto

# Las mismas restricciones que el contrato Pydantic del Mini-TP 1
RANGOS = {"sucursal_id": (1, 9), "sku_id": (1, 96), "dia_semana": (0, 6), "mes": (1, 12)}
NO_NEGATIVOS = ("demanda_lag_1", "demanda_lag_7", "media_movil_7")


def validar(f) -> list[str]:
    errores = [f"{k} fuera de rango {lo}..{hi}: {getattr(f, k)}"
               for k, (lo, hi) in RANGOS.items() if not lo <= getattr(f, k) <= hi]
    if f.precio_unitario <= 0:
        errores.append(f"precio_unitario debe ser > 0: {f.precio_unitario}")
    errores += [f"{k} debe ser >= 0: {getattr(f, k)}" for k in NO_NEGATIVOS if getattr(f, k) < 0]
    return errores


def a_vector(f) -> list[float]:
    return [float(getattr(f, k)) for k in FEATURES]      # mismo orden que en el entrenamiento


def respuesta(y: float, indice: int = 0):
    return scoring_pb2.Prediction(demanda_estimada=round(float(y), 2), unidad="unidades/día",
                                  modelo_nombre=MODEL_NAME, modelo_version=MODEL_VERSION,
                                  indice=indice)


class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    def Predict(self, request, context):
        errores = validar(request)
        if errores:      # el equivalente gRPC del 422 de FastAPI
            context.abort(grpc.StatusCode.INVALID_ARGUMENT, "; ".join(errores))
        y = modelo.predict([a_vector(request)])[0]
        return respuesta(y)

    def PredictLote(self, request, context):
        casos = list(request.casos)
        for i, c in enumerate(casos):
            errores = validar(c)
            if errores:
                context.abort(grpc.StatusCode.INVALID_ARGUMENT, f"caso {i}: " + "; ".join(errores))
        # Puntuamos por tandas: una sola llamada al modelo por tanda (vectorizado) y vamos
        # emitiendo cada predicción apenas está lista, sin esperar a terminar todo el lote.
        TANDA = 64
        for ini in range(0, len(casos), TANDA):
            if not context.is_active():          # el cliente cortó: no seguimos puntuando
                return
            X = [a_vector(c) for c in casos[ini:ini + TANDA]]
            for j, y in enumerate(modelo.predict(X)):
                yield respuesta(y, indice=ini + j)

    def Health(self, request, context):
        return scoring_pb2.HealthResponse(status="ok", modelo_cargado=modelo is not None,
                                          modelo_nombre=MODEL_NAME, modelo_version=MODEL_VERSION,
                                          n_features=N_FEATURES)


try:                     # si la celda se corre dos veces, liberar el puerto
    servidor.stop(0)
except NameError:
    pass
servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50051")
servidor.start()
print("Servidor gRPC en localhost:50051")


Servidor gRPC en localhost:50051


## 5. El cliente (unary)

In [ ]:
canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

h = stub.Health(scoring_pb2.HealthRequest())
print(f"health: {h.status} | {h.modelo_nombre} v{h.modelo_version} | n_features={h.n_features}")

# Un caso válido: el mismo CASO_VALIDO del cliente REST del Mini-TP 1
CASO_VALIDO = dict(sucursal_id=3, sku_id=42, dia_semana=5, mes=7, es_finde=True, es_feriado=False,
                   en_promocion=True, precio_unitario=1800.0, demanda_lag_1=52.0,
                   demanda_lag_7=60.0, media_movil_7=55.0)
entrada = scoring_pb2.Features(**CASO_VALIDO)
r = stub.Predict(entrada)
print(f"predicción: {r.demanda_estimada} {r.unidad} | modelo: {r.modelo_nombre} v{r.modelo_version}")

# Un caso inválido: sucursal fuera de rango y precio negativo (el 422 de la Sesión 1)
try:
    stub.Predict(scoring_pb2.Features(**{**CASO_VALIDO, "sucursal_id": 50, "precio_unitario": -100.0}))
except grpc.RpcError as e:
    print(f"caso inválido -> {e.code().name}: {e.details()}")

# El contrato tipado también rechaza del lado del cliente lo que no encaja
try:
    scoring_pb2.Features(sucursal_id="tres")
except TypeError as e:
    print(f"tipo incorrecto -> rechazado antes de enviar: {e}")


health: ok | gout-demanda-rf v1.0.0 | n_features=11
predicción: 87.74 unidades/día | modelo: gout-demanda-rf v1.0.0
caso inválido -> INVALID_ARGUMENT: sucursal_id fuera de rango 1..9: 50; precio_unitario debe ser > 0: -100.0
tipo incorrecto -> rechazado antes de enviar: 'str' object cannot be interpreted as an integer


## 6. Streaming

`PredictLote` recibe un lote de casos (una semana de un SKU en las 9 sucursales, 63 casos) y devuelve
una predicción por caso en un stream. El cliente las procesa a medida que llegan.


In [ ]:
# Un lote realista: el SKU 42 en las 9 sucursales durante una semana de julio (9 x 7 = 63 casos)
lote = scoring_pb2.Lote(casos=[
    scoring_pb2.Features(sucursal_id=s, sku_id=42, dia_semana=d, mes=7, es_finde=d >= 5,
                         es_feriado=False, en_promocion=(d == 4), precio_unitario=1800.0,
                         demanda_lag_1=50.0 + 2 * s, demanda_lag_7=55.0 + s, media_movil_7=53.0 + 1.5 * s)
    for s in range(1, 10) for d in range(7)])

recibidas = []
for p in stub.PredictLote(lote):
    recibidas.append(p)
    if p.indice < 3:
        print(f"  llegó caso {p.indice}: {p.demanda_estimada} {p.unidad}")
print(f"  ... {len(recibidas)} predicciones recibidas por stream (lote de {len(lote.casos)})")

# Resumen: demanda semanal estimada por sucursal
por_suc = np.array([p.demanda_estimada for p in sorted(recibidas, key=lambda p: p.indice)]).reshape(9, 7)
for s, fila in enumerate(por_suc, start=1):
    print(f"  sucursal {s}: {fila.sum():6.1f} unidades/semana | pico {fila.max():.1f} (día {fila.argmax()})")

# Una entrada inválida en el lote corta el stream con INVALID_ARGUMENT
try:
    malo = scoring_pb2.Lote(casos=[entrada, scoring_pb2.Features(**{**CASO_VALIDO, "mes": 13})])
    list(stub.PredictLote(malo))
except grpc.RpcError as e:
    print(f"lote con un caso inválido -> {e.code().name}: {e.details()}")


  llegó caso 0: 62.35 unidades/día
  llegó caso 1: 62.35 unidades/día
  llegó caso 2: 62.35 unidades/día
  ... 63 predicciones recibidas por stream (lote de 63)
  sucursal 1:  474.0 unidades/semana | pico 75.7 (día 5)
  sucursal 2:  484.7 unidades/semana | pico 78.7 (día 5)
  sucursal 3:  488.7 unidades/semana | pico 79.5 (día 5)
  sucursal 4:  504.4 unidades/semana | pico 82.0 (día 5)
  sucursal 5:  526.2 unidades/semana | pico 83.6 (día 4)
  sucursal 6:  535.4 unidades/semana | pico 84.8 (día 4)
  sucursal 7:  538.1 unidades/semana | pico 84.8 (día 4)
  sucursal 8:  547.8 unidades/semana | pico 86.2 (día 5)
  sucursal 9:  554.7 unidades/semana | pico 87.3 (día 5)
lote con un caso inválido -> INVALID_ARGUMENT: caso 1: mes fuera de rango 1..12: 13


## 7. Comparación de latencia gRPC vs REST

Levanto el endpoint REST de la Sesión 1 dentro del notebook: el mismo `/v1/predict` con el mismo
contrato Pydantic (`extra="forbid"`) y el **mismo objeto `modelo`** en memoria, así la única diferencia
entre las dos mediciones es el protocolo. (En Colab no se puede apuntar a una API que corre en tu
compu: por eso el REST se levanta acá, con el mismo código del Mini-TP 1.)

Medición (N = 500 llamadas por fila, después de un warm-up de 20):
- **REST** con `requests.post` (una conexión nueva por llamada, como el tutorial) y con
  `requests.Session` (conexión reutilizada, la comparación justa con gRPC).
- **gRPC** unary, reutilizando el canal.
- **Sólo el modelo**, sin red, para saber cuánto de la latencia es inferencia.
- Un **lote de 63 casos**: 63 llamadas REST contra un stream gRPC.


In [ ]:
import threading, requests, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field

# --- El REST de la Sesión 1 (mismo contrato Pydantic del Mini-TP 1) ---
class PrediccionRequest(BaseModel):
    sucursal_id: int = Field(..., ge=1, le=9)
    sku_id: int = Field(..., ge=1, le=96)
    dia_semana: int = Field(..., ge=0, le=6)
    mes: int = Field(..., ge=1, le=12)
    es_finde: bool
    es_feriado: bool
    en_promocion: bool
    precio_unitario: float = Field(..., gt=0)
    demanda_lag_1: float = Field(..., ge=0)
    demanda_lag_7: float = Field(..., ge=0)
    media_movil_7: float = Field(..., ge=0)
    model_config = {"extra": "forbid"}


class PrediccionResponse(BaseModel):
    demanda_estimada: float
    unidad: str = "unidades/día"
    modelo_nombre: str
    modelo_version: str


api = FastAPI(title="Gout - Forecasting de demanda (REST, Sesión 1)")


@api.post("/v1/predict", response_model=PrediccionResponse)
def predict(req: PrediccionRequest):
    x = [float(getattr(req, k)) for k in FEATURES]
    y = modelo.predict([x])[0]
    return PrediccionResponse(demanda_estimada=round(float(y), 2),
                              modelo_nombre=MODEL_NAME, modelo_version=MODEL_VERSION)


REST_PORT = 8001
REST_URL = f"http://127.0.0.1:{REST_PORT}/v1/predict"


def _rest_arriba() -> bool:
    try:
        return requests.post(REST_URL, json=CASO_VALIDO, timeout=1).status_code == 200
    except requests.ConnectionError:
        return False


if not _rest_arriba():
    threading.Thread(target=lambda: uvicorn.run(api, host="127.0.0.1", port=REST_PORT, log_level="error"),
                     daemon=True).start()
    for _ in range(50):
        time.sleep(0.2)
        if _rest_arriba():
            break
print("REST:", requests.post(REST_URL, json=CASO_VALIDO).json())
print("REST inválido:", requests.post(REST_URL, json={**CASO_VALIDO, "sucursal_id": 50}).status_code)


REST: {'demanda_estimada': 87.74, 'unidad': 'unidades/día', 'modelo_nombre': 'gout-demanda-rf', 'modelo_version': '1.0.0'}
REST inválido: 422


In [ ]:
import statistics

N, WARMUP = 500, 20
x_modelo = [[float(CASO_VALIDO[k]) for k in FEATURES]]
sesion = requests.Session()


def medir(fn, n=N):
    """Latencia por llamada en ms: mediana y p95 sobre n llamadas, después de un warm-up."""
    for _ in range(WARMUP):
        fn()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort()
    return statistics.median(ts), ts[int(0.95 * len(ts)) - 1], statistics.mean(ts)


res = {
    "solo el modelo (sin red)": medir(lambda: modelo.predict(x_modelo)),
    "gRPC unary": medir(lambda: stub.Predict(entrada)),
    "REST (requests.Session)": medir(lambda: sesion.post(REST_URL, json=CASO_VALIDO)),
    "REST (requests.post)": medir(lambda: requests.post(REST_URL, json=CASO_VALIDO)),
}
m_modelo, m_grpc = res["solo el modelo (sin red)"][0], res["gRPC unary"][0]

print(f"{'':<27}{'mediana':>9}{'p95':>9}{'media':>9}   overhead vs modelo")
for k, (med, p95, media) in res.items():
    extra = "" if k.startswith("solo") else f"{med - m_modelo:8.3f} ms"
    print(f"{k:<27}{med:>7.3f}ms{p95:>7.3f}ms{media:>7.3f}ms   {extra}")

for k in ("REST (requests.Session)", "REST (requests.post)"):
    print(f"gRPC es {res[k][0] / m_grpc:.1f}x más rápido que {k} (mediana)")

# Tamaño en el cable del mismo caso
import json as _json
print(f"\nbytes por request: protobuf {len(entrada.SerializeToString())} B | JSON {len(_json.dumps(CASO_VALIDO).encode())} B")

# --- Un lote de 63 casos: 63 llamadas REST vs un stream gRPC ---
casos_json = [{k: (bool(getattr(c, k)) if k in ('es_finde', 'es_feriado', 'en_promocion') else getattr(c, k))
               for k in FEATURES} for c in lote.casos]
t0 = time.perf_counter(); [sesion.post(REST_URL, json=c) for c in casos_json]; lote_rest = (time.perf_counter() - t0) * 1000
t0 = time.perf_counter(); list(stub.PredictLote(lote)); lote_grpc = (time.perf_counter() - t0) * 1000
print(f"\nlote de {len(casos_json)} casos: REST {lote_rest:.1f} ms ({len(casos_json)} llamadas) | "
      f"gRPC stream {lote_grpc:.1f} ms (1 llamada) -> {lote_rest / lote_grpc:.1f}x")


                             mediana      p95    media   overhead vs modelo
solo el modelo (sin red)    10.961ms 19.964ms 13.481ms   
gRPC unary                  11.683ms 15.766ms 12.351ms      0.722 ms
REST (requests.Session)     13.958ms 22.341ms 15.859ms      2.997 ms
REST (requests.post)        14.706ms 21.132ms 15.659ms      3.745 ms
gRPC es 1.2x más rápido que REST (requests.Session) (mediana)
gRPC es 1.3x más rápido que REST (requests.post) (mediana)

bytes por request: protobuf 48 B | JSON 216 B

lote de 63 casos: REST 1249.3 ms (63 llamadas) | gRPC stream 40.3 ms (1 llamada) -> 31.0x


### Reflexión

**Latencia.** Por llamada, gRPC le gana a REST por poco (1.2x–1.3x en la mediana: 11.7 ms contra 14.0–14.7 ms), porque con este
modelo la latencia la domina la **inferencia**: el Random Forest de 120 árboles tarda ≈11 ms por
predicción, y el protocolo es lo de menos. Donde sí se ve la diferencia es en el **overhead** que agrega
cada protocolo por encima del modelo: gRPC suma 0.7 ms y REST entre 3.0 y 3.7 ms, o sea 4x–5x menos overhead (binario +
HTTP/2 + conexión reutilizada), y el mismo caso pesa 48 B en protobuf contra 216 B en JSON. La diferencia
grande aparece con el lote: 63 llamadas REST tardaron 1249 ms y **un** stream gRPC 40 ms, 31x más rápido. Parte de eso
es protocolo (una sola conexión y un solo request), pero la mayor parte es que el servidor puntúa el
lote vectorizado en vez de llamar al modelo 63 veces.

**Cuándo usaría cada uno.** gRPC para lo **interno**: el servicio de pronóstico llamado por el
planificador de producción o por el pipeline que puntúa las 9 sucursales × 96 SKUs todas las noches,
donde importan el volumen, el streaming y un contrato estricto entre equipos. REST en el **borde**:
la app o el dashboard de las sucursales, integraciones con terceros y cualquier cosa que se pruebe
con `curl` o se consuma desde el navegador.

**El costo de gRPC.** Un contrato que hay que versionar y compilar en cliente y servidor (cambiar el
`.proto` obliga a regenerar los stubs de los dos lados), menos tooling para depurar (no se lee con
`curl` ni en el navegador), el navegador no lo habla directo (hace falta gRPC-Web y un proxy), y la
validación de rangos no viene gratis como con Pydantic: la tuve que escribir a mano en el servidor.


In [ ]:
# limpieza
canal.close(); servidor.stop(0); sesion.close()
print("Servidor gRPC detenido (el REST corre en un hilo daemon y termina con el kernel).")


Servidor gRPC detenido (el REST corre en un hilo daemon y termina con el kernel).
